In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations: convert images to tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),                # converts PIL image to tensor [0,1]
    transforms.Normalize((0.5,), (0.5,)) # normalize to [-1,1]
])

In [4]:
# Download and load training dataset
train_dataset = datasets.MNIST(root='data_RNN', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:03<00:00, 2.96MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 119kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 1.10MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 640kB/s]


In [5]:
# Download and load test dataset
test_dataset = datasets.MNIST(root='data_RNN', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self,input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size 
        self.num_layers=num_layers
        # RNN Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers, batch_first=True)
        self.fc=nn.Linear(hidden_size, 10)
    def forward(self,x):
        h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out,_=self.rnn(x,h0)
        out=self.fc(out[:, -1 ,:])

        return out
        
        
        
        
        
        
        

In [10]:
import torch.nn as nn
import torch.optim as optim
input_size=28

model=RNN(input_size)
criterion = nn.CrossEntropyLoss()
optim=optim.Adam(model.parameters()) 

In [13]:
import torch
# Training
epochs=10
for epoch in range(epochs):
    model.train() 
    for xb,yb in train_loader:
        optim.zero_grad()
        xb=xb.squeeze(1) #add singleton direction, since RNN expects 3D outputs
        output=model(xb) 
        loss=criterion(output.squeeze(),yb) # output.shape = (64, 1) but yb.shape = (64), they dont match
        # that us why we used squeeze
        loss.backward()
        optim.step()
    print(f" epoch={epoch+1}/{epochs} and loss ={loss.item()}")

 epoch=1/10 and loss =0.26818743348121643
 epoch=2/10 and loss =0.6173266172409058
 epoch=3/10 and loss =0.1823180466890335
 epoch=4/10 and loss =0.04627590626478195
 epoch=5/10 and loss =0.10176907479763031
 epoch=6/10 and loss =0.012563595548272133
 epoch=7/10 and loss =0.22435475885868073
 epoch=8/10 and loss =0.19761770963668823
 epoch=9/10 and loss =0.01859014481306076
 epoch=10/10 and loss =0.18164779245853424


In [16]:
# Evaluate
model.eval()
with torch.no_grad():
    correct=0
    total=0
    for xb,yb in test_loader:
        xb=xb.squeeze(1)
        output=model(xb)
        predicted = torch.argmax(output, dim=1)        
        total+=yb.size(0)
        correct += (predicted==yb).sum().item()

    print(f"Accuracy = {correct/total *100}") 
        
        

Accuracy = 96.55
